In [4]:
import pandas as pd

In [5]:
df = pd.read_excel("../data/raw/eventos_chubut.xlsx")

In [6]:
df.columns.tolist()

['noticia_id',
 'link',
 'fecha_publicacion',
 'titulo',
 'localidad',
 'actores_participantes',
 'demanda_principal',
 'tipo_accion',
 'tipo_accion_detalle',
 'disrupcion',
 'escala',
 'intensidad',
 'respuesta_estatal',
 'presencia_policial',
 'accionar_policial',
 'tipo_intervencion_estatal',
 'uso_fuerza',
 'heridos',
 'detenciones',
 'judicializacion',
 'coord_interinstitucional',
 'canales_informales',
 'amenaza_politica',
 'resultado_episodio',
 'fuente',
 'observaciones',
 'notas_estandarizacion']

In [7]:
df_guardar = df.copy()

In [8]:
df_guardar["actores_participantes"].unique().tolist()

['Sindicato de la Salud Pública',
 'No aplica',
 'No especificado',
 'Auxiliares de la educación',
 'Miembros de comunidades originarias',
 'Personal de la salud/Sisap',
 'Docentes',
 'Sisap',
 'Consejo de Bienestar Policial',
 'amigos/colegas/seguidores de su programa radial/bandas musicales/alumnos/vecinos de distintas organizaciones sociales',
 'Error',
 'Sindicato de Camioneros',
 'Trabajadores del servicio de recolección de residuos y de la Planta de Tratamiento',
 'Feriantes del barrio Perón',
 'Soeme',
 'Consejo de Bienestar Policial del Chubut',
 'Consejo de Bienestar Policial/Familia policial',
 'Estibadores',
 'Sindicato de la Salud Pública (Sisap)',
 'Policías/Familias de policías',
 'Policía',
 'Familia de Maximiliano Curiqueo',
 'Policía activo y pasivo',
 'Vecinos de la Meseta del Chubut/Intendentes de la región/Referentes sociales y de comunidades originarias/Representantes gremiales y trabajadores de las seccionales Puerto Madryn/Paso de Indios y Esquel de la Unión Obre

In [10]:
df_guardar = df_guardar[~df_guardar["actores_participantes"].isin(["Error", "No aplica"])].copy()

Debido a la cantidad de actores y repitencias de nombres por siglas, mayusculas y minusculas se decidio hacer una limpieza para unificar todo

In [13]:
reemplazos = {
    # --- Sindicatos / siglas ---
    "sindicato de la salud pública": "Sindicato de la Salud Pública (Sisap)",
    "sindicato de la salud pública (sisap)": "Sindicato de la Salud Pública (Sisap)",
    "sindicato de la salud pública (susap)": "Sindicato de la Salud Pública (Sisap)",
    "sisap": "Sindicato de la Salud Pública (Sisap)",
    "personal de la salud/sisap": "Sindicato de la Salud Pública (Sisap)",
    "trabajadores de la salud": "Sindicato de la Salud Pública (Sisap)",
    "consejo de bienestar policial": "Consejo de Bienestar Policial",
    "consejo de bienestar policial del chubut": "Consejo de Bienestar Policial",
    "consejo de bienestar policial/familia policial": "Consejo de Bienestar Policial",
    "soeme": "SOEME",
    "ema": "EMA",
    "ema/otros sindicatos": "EMA",
    "hompanera/ate": "ATE",
    "unión tranviarios automotor (uta)": "UTA",
    "choferes de la unión tranviarios automotor (uta)": "UTA",

    # --- Policía ---
    "policía": "Policía",
    "policía activo y pasivo": "Policía",
    "policías/familias de policías": "Policía",
    "familia policial": "Policía",

    # --- Antimineros / asambleístas / ambientalistas ---
    "antimineros": "Antimineros",
    "grupo de manifestantes antimineros": "Antimineros",
    "grupos antimineros": "Antimineros",
    "grupos antimineros/manifestantes": "Antimineros",
    "manifestantes antimineros": "Antimineros",
    "protestantes antiminera": "Antimineros",
    "manifestantes contra la minería": "Antimineros",
    "vecinos de la asamblea del no a la mina": "Asamblea No a la Mina",
    "vecinos del 'no a la mina'/ambientalistas": "Asamblea No a la Mina",
    "asambleístas": "Asambleístas",
    "asambleistas": "Asambleístas",
    "asambleístas ": "Asambleístas",
    "ambientalistas": "Ambientalistas",

    # --- Manifestantes / vecinos genéricos ---
    "manifestantes": "Manifestantes",
    "grupo de manifestantes": "Manifestantes",
    "integrantes de una marcha": "Manifestantes",
    "manifestantes ": "Manifestantes",
    "vecinos": "Vecinos",
    "vecinos y vecinas": "Vecinos",
    "vecinos autoconvocados": "Vecinos",
    "chubutenses": "Vecinos",
    "vecinos en emergencia": "Vecinos",
    "rodanteros": "Rodanteros",
    "dueños de casillas rodantes": "Rodanteros",

    # --- Mapuches / originarios ---
    "mapuches": "Comunidades originarias",
    "miembros de comunidades originarias": "Comunidades originarias",
    "representantes de comunidades originarias/otras organizaciones": "Comunidades originarias",
    "activistas mapuches": "Comunidades originarias",

    # --- Docentes / educación ---
    "docentes": "Docentes",
    "auxiliares de la educación": "Auxiliares de la educación",
    "padres y madres de 6to grado de la escuela 15": "Padres Escuela 15",

    # --- Trabajadores varios ---
    "trabajadores": "Trabajadores",
    "trabajadores pesqueros": "Trabajadores pesqueros",
    "trabajadores autoconvocados de la pesca/vecinos": "Trabajadores pesqueros",
    "trabajadores de la cereza": "Trabajadores de la cereza",
    "operarios del cose": "Operarios del COSE",
    "trabajadores y trabajadoras del área de espacios verdes": "Trabajadores de Espacios Verdes",
    "trabajadores del servicio de recolección de residuos y de la planta de tratamiento": "Trabajadores de residuos",
    "trabajadores de la uocra/vecinos": "UOCRA",

    # --- Movimientos sociales / políticos ---
    "movimientos sociales": "Movimientos sociales",
    "militantes sociales": "Movimientos sociales",
    "militantes de un sector político": "Militantes políticos",
    "chubut al frente/pj-frente patriótico chubutense/por trelew": "Frentes políticos",
    "madernistas": "Madernistas",
    "intendente de trelew / fieles seguidores": "Intendente de Trelew",
    "intendente de rawson / vecinos capitalinos": "Intendente de Rawson",

    # --- Vándalos / encapuchados ---
    "vándalos": "Vándalos",
    "encapuchados": "Encapuchados",
    "gente encapuchada": "Encapuchados",
    "turba de exaltados/revoltosos": "Vándalos",
    "grupo reducido de inadaptados/movimiento de trabajadores excluidos": "Movimientos sociales",
    "imputados": "Imputados",

    # --- Otros ---
    "no aplica": "No aplica",
    "no especificado": "No especificado",
    "error": "Error",
    "feriantes del barrio perón": "Feriantes del barrio Perón",
    "estibadores": "Estibadores",
    "traumatólogos": "Traumatólogos",
    "cuidadoras de adultos mayores": "Cuidadoras de adultos mayores",
    "empleados del concejo deliberante": "Empleados del Concejo Deliberante",
    "asociación de magistratura y funcionariado judicial del chubut": "Magistratura judicial",
    "sindicato de camioneros": "Camioneros",
    "familia de maximiliano curiqueo": "Familia Curiqueo",
    "familiares/organizaciones sociales": "Familiares y organizaciones sociales",
    "mamá cultiva/cannabis medicinal terapéutico puerto madryn/cepa cannabicultores del este patagónico": "Cannabicultores",
    "multisectorial vivas y libres nos queremos": "Multisectorial Vivas y Libres",
    "vecinos de la meseta del chubut/intendentes de la región/referentes sociales y de comunidades originarias/representantes gremiales y trabajadores de las seccionales puerto madryn/paso de indios y esquel de la unión obrera de la construcción (uocra)": "Multiactor (meseta Chubut)",

    # --- Aliados políticos de los detenidos ---
    "dirigentes políticos y sociales / familias de los detenidos / activistas / movimiento de trabajadores excluidos / concejalas lorena alcalá, carol williams, claudia iun, mariela flores torres (mc) / diputados madernistas leila lloyd jones, josé giménez y angel chiquichano": "Multiactor (dirigentes y familias)",
}

In [14]:
df_guardar["actores_norm"] = df_guardar["actores_participantes"].str.strip().str.lower()

# Aplicamos el mapeo; si no está en el diccionario, dejamos el original
df_guardar["actores_limpio"] = df_guardar["actores_norm"].map(reemplazos).fillna(df_guardar["actores_participantes"])

# Chequeo
print("Antes:", df_guardar["actores_participantes"].nunique())
print("Después:", df_guardar["actores_limpio"].nunique())
print(df_guardar["actores_limpio"].value_counts())

Antes: 86
Después: 51
actores_limpio
No especificado                                                                                                          18
Vecinos                                                                                                                   9
Sindicato de la Salud Pública (Sisap)                                                                                     8
Manifestantes                                                                                                             7
Antimineros                                                                                                               7
Asambleístas                                                                                                              7
Comunidades originarias                                                                                                   5
Consejo de Bienestar Policial                                                                  

In [16]:
df_guardar.to_excel("../data/processed/eventos_chubut_limpio.xlsx", index=False)

In [56]:
df_guardar['localidad'].unique().tolist()

['Chubut',
 'No especificado',
 'Rawson',
 'Esquel',
 'Trelew',
 'Comodoro Rivadavia',
 'Puerto Madryn',
 'Puerto Rawson',
 'Gastre',
 'El Bolsón',
 'Ushuaia',
 'Los Aromos',
 'Paso de Indios',
 'Madryn',
 'Gobernador Costa',
 'Cholila',
 'Ate',
 'Parque Nacional Los Alerces',
 'El Chubut']

In [57]:
reemplazos_loc = {
    # Normalizar duplicados
    "madryn": "Puerto Madryn",
    "puerto rawson": "Rawson"
}

df_guardar["localidad"] = df_guardar["localidad"].replace(reemplazos_loc)

In [58]:
df_guardar["tipo_accion"].unique().tolist()

['Paro',
 'No especificado',
 'Corte',
 'Toma de edificio',
 'Mixta',
 'Marcha',
 'Manifestación',
 'Asamblea',
 'Otra',
 'Acto',
 'Caravana',
 'Escrache',
 'Destrozos',
 'Difusión de comunicado']

In [59]:
df_guardar["demanda_principal"].unique().tolist()

['Pago de deuda de paritarias y retroactivos',
 'No especificado',
 'falta de convocatoria a paritarias salariales',
 'solución a la titularidad de tierras',
 'Pago de deuda paritaria y mejoras salariales y laborales',
 'Demandas salariales y edilicias',
 'Recomposición salarial y condiciones laborales de los empleados de la salud pública',
 'Reclamo salarial',
 'justicia por el femicidio de la profesora Daniela Velasco',
 'Despedida de 30 trabajadores de la firma GP Inversiones',
 'Pago del aumento del adicional por insalubridad',
 'Exigir pago de 1.000 pesos y costos de espacio de 1.500-2.500 pesos, y que la organización civil no cobre sin proyecto',
 'Pago de las recategorizaciones acordadas meses atrás',
 'Salario digno para el personal policial',
 'Mejoras salariales para uniformados activos y pasivos',
 'Bloquear descarga de langostinos',
 'Reclamo de pago de paritarias salariales y mejora de condiciones laborales en el sector salud',
 'Mejoras salariales y laborales para policía